# Train two-tower encoder-decoder classifier - StereoQueerEval Task A

Binary stereotype detection (`yes` / `no`) with `src/train_xattn.py`.

The video (title + description) is the **encoder memory**; the comment is the
**decoder input**. Each decoder block refines the comment against itself
(self-attention, full - not causal), then attends into the video memory
(cross-attention). No generation: pool the comment side and read a 2-way logit.

Design here: MiniLM encoder + **stacked decoder blocks** (warm-started from the
encoder by default), a **gated video-fusion** head (default `--fusion gated`), with
an optional **two-phase** schedule (keep the encoder frozen first, then unfreeze the
top blocks once the head has settled).

- Encoder: `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2` (384-d)
- Video budget 256 tokens: all videos fit (description max measured 216)
- Same video-level split / labels / metrics as `train.py`
- Frozen phase => the video K/V is deterministic (but **not** cached: nothing
  is memoised, see `src/report_cost.py` for the parameter cost)

Works on **Kaggle** and Colab.


In [ ]:
import os, subprocess

if not os.path.isdir('Try'):
    subprocess.run('git clone https://github.com/AmnO-O/Try.git', shell=True, check=True)
os.chdir('Try')
subprocess.run('pip install -q -r requirements.txt', shell=True, check=True)
print('cwd:', os.getcwd())
print('src has train_xattn.py:', os.path.exists('src/train_xattn.py'))


## 1. Locate the data

The TSVs are not committed. This cell finds them automatically (Kaggle input,
`/content`, cwd, `./LGBT`) and exports `LGBT_DATA_DIR`.


In [ ]:
import glob, os

candidates = []
if 'LGBT_DATA_DIR' in os.environ and os.path.isdir(os.environ['LGBT_DATA_DIR']):
    candidates.append(os.environ['LGBT_DATA_DIR'])
if 'KAGGLE_INPUT_PATH' in os.environ and os.path.isdir(os.environ['KAGGLE_INPUT_PATH']):
    candidates.append(os.environ['KAGGLE_INPUT_PATH'])
for root in ('/kaggle/input', '/content', os.path.abspath('.')):
    candidates.append(root)
if os.path.isdir('LGBT'):
    candidates.append(os.path.abspath('LGBT'))

def _first(roots):
    for root in roots:
        hits = sorted(glob.glob(os.path.join(root, '**', 'StereoQueerEval_*_training.tsv'), recursive=True))
        if hits:
            return hits
    return []

hits = _first(candidates)
if not hits:
    raise RuntimeError('StereoQueerEval_*_training.tsv not found - mount the dataset first')
os.environ['LGBT_DATA_DIR'] = os.path.dirname(hits[0])
os.environ['LGBT_WEIGHTS_DIR'] = os.path.abspath('weights')
os.makedirs(os.environ['LGBT_WEIGHTS_DIR'], exist_ok=True)
print('data dir :', os.environ['LGBT_DATA_DIR'])
print('weights  :', os.environ['LGBT_WEIGHTS_DIR'])
print('files    :', [os.path.basename(p) for p in hits])
assert len(hits) >= 1


## 2. Sanity check the split

Row counts, label balance and the video-grouped train/val split. Only **EN / IT /
NL** are loaded - the FA files stay held out (if `StereoQueerEval_FA_*` is present
in the data dir, loading *all* languages would leak it into train and val).
Defaults are `val-fraction=0.1`, `split-seed=42` - the same ones the cells below use.


In [ ]:
import sys
sys.path.insert(0, os.path.abspath('src'))
import data as data_mod
from config import MAX_LEN

LANGS = ['EN', 'IT', 'NL']
df = data_mod.load_frame(languages=LANGS, max_len=MAX_LEN)
print('rows:', len(df), '| by lang:', df['lang'].value_counts().to_dict())
tr, va = data_mod.split_by_video(df, 0.1, 42)
print('train/val:', len(tr), '/', len(va))
print('video overlap:', len(set(tr['yt_title']) & set(va['yt_title'])))


## 3. Train the encoder-decoder model

Each tower has its **own MiniLM encoder** (the default since the separate-encoder
change), topped by **3 decoder blocks** (self-attn on the comment, then
cross-attn into the video; 4 heads) with light
regularization (dropout 0.1, weight decay 0.01) and the top **2 encoder blocks of
each tower unfrozen**.

Three parameter groups, because the three parts want very different step sizes:

| group | LR | why |
|---|---|---|
| backbone | 2e-5 | pretrained, a large step destroys it |
| decoder blocks | 1e-4 (`--xattn-lr`) | separated from the head because of the step-size mismatch below |
| head | 1e-3 | `LayerNorm + Linear(d, 2)` plus the pooling query |

The blocks used to be lumped in with the head, i.e. the whole decoder stack was
trained at 1e-3. With AdamW the per-step move is ~LR whatever the gradient scale,
and `clip_grad_norm_` does **not** shrink it (Adam renormalises per coordinate), so
a block at the head LR was moving ~5% of its own weight scale per step. Warm-start
(below) means the blocks no longer start from random init, but they are still trained
as their own parameter group. `--xattn-lr 1e-3` restores the old behaviour for
comparison.

Frozen encoder blocks are switched to `eval()`, so their Dropout is off and the
frozen half of each tower is deterministic while training (`--keep-frozen-dropout`
restores the stochastic version). That invariant is enforced by
`XAttnClassifier.train()` itself, not by the loop: `nn.Module.train()` walks every
child and re-enables the frozen blocks' Dropout, so the model re-applies
`eval_frozen_modules()` on the way out. Previously the loop did it as a second
line, which meant any other entry point got a stochastic frozen tower.

It also used to *fail silently* for a fully frozen encoder (`--unfreeze-layers 0`,
and phase 1 of `--freeze-epochs`): the ancestor test in `eval_frozen_modules`
compared a top-level module name against the name itself, so the whole encoder was
skipped and all 36 of its Dropouts stayed in train mode. With `unfreeze_layers >= 1`
(what this notebook trains, and what section 7's baseline uses) it already worked.
Section 6 marks the row that is affected.

Masked attention also has a hard requirement: **every row needs at least one real
token.** The comment pooling fills the padded scores with `-inf` and then takes a
softmax, and a row with no finite score is `0/0`. The damage is not limited to the
forward value - softmax's backward multiplies by its own NaN output, so the
*gradient* is NaN too and the run dies on the first step containing such a row.
`PairedDataset` rejects an all-zero mask at load time with the offending row and
its text, the pooling is written so that a row that slipped through anyway gets a
zero distribution instead of a NaN, and the model boundary rejects an all-padded
*video* row up front (`_validate_video_mask`), because the `need_weights=True`
diagnostic path is NaN there. `[CLS]`/`[SEP]` make this unreachable for a real
tokenizer (an empty string still yields two tokens), so the guard is a safety net
rather than a fix for an observed failure - `src/probe_allpad.py` is the evidence for
both halves.

**How the video reaches the head** is set by `--fusion`, and the default is now
**`gated`**:

| `--fusion` | head input | channels from the video tower |
|---|---|---|
| `none` | `pooled_c` | cross-attention only (the previous default) |
| `concat` | `[pooled_c ; mean-pool(H_v)]` | cross-attention **and** a parallel late-fusion term (the old `--pooled-v`) |
| `gated` | `pooled_c + sigmoid(W.[pooled_c ; mean-pool(H_v)]) * video_proj(mean-pool(H_v))` | cross-attention plus a gated residual on that same parallel term |

The `gated` gate bias is initialised to `-1.5` (`sigmoid ~ 0.18`), so it starts
mostly closed and training has to open it. Section 8 measures all three; a single
seed-42 pair is not evidence (the run-to-run spread is ~1pt).

The decoder blocks are **warm-started** from the top encoder layers by default
(`warm_start_decoder_from_backbone`: self-attn, cross-attn, FFN and LayerNorm), so
they no longer begin from random init; `--no-warm-start` restores random init and is
the control for that choice. Every command here states `--fusion` and `--warm-start`
explicitly so a future default change cannot silently alter what a row means.

What `--unfreeze-layers n` actually re-opens is worth checking, because it is
name-based (`layer.N` BERT-style, `layers.N` ModernBERT) and the two backbones do
not have the same top edge. `python src/probe_unfreeze.py` derives the expected
set from the *module tree* and compares: 12 blocks for MiniLM, 22 for mmBERT, and
at n = 0, 1, 2 and all the trainable set is exactly the last n blocks - plus the
encoder's output normalisation, where the architecture keeps it outside the block
stack. ModernBERT has `final_norm` as a sibling of `layers`; BERT folds the
equivalent `layer.N.output.LayerNorm` *inside* block N. Training "the last n
blocks" would therefore train the top of a BERT tower and leave the top of a
ModernBERT tower frozen - a silent difference in exactly the backbone comparison
this model exists to make. That norm is now unfrozen with the last block, while
the embeddings stay frozen at every n (the largest table, and the one
`run_setfit.py` needs free of `requires_grad`). `n < 0` and `n >` the real block
count raise instead of silently freezing / unfreezing everything: on a 2-layer
model `--unfreeze-layers 5` used to mean "all".

**Nothing is cached.** `forward()` runs both towers every step, so a repeated
video is re-encoded; a shared encoder does not change that by itself, because a
cache also needs the *video* weights to be stationary - true while fully frozen,
and at inference, but not across optimizer steps of a trainable video tower. The
docstring used to claim a shared encoder "makes the video K/V cacheable per
video"; that was ahead of the code and is removed. Nor is the cost uniform
across backbones: 240.6M separate / 122.98M shared for MiniLM, but ~635M / ~328M
for mmBERT (`python src/report_cost.py`). Separate encoders are an experimental
variable, not a free default.

The attention pool **starts uniform**. `python src/probe_pool_init.py` measures
the initial entropy against several query scales: at the default 0.02 the
comment-pooling distribution is at the uniform maximum (`H / log L = 1.0000`,
max weight 1/L, i.e. mean pooling) and only sharpens well above 0.5 (1.0000 ->
0.9814 at 1.0, max weight 0.0618 -> 0.1030). That is an init property, not a
bug - the query trains - but "the pool learned to select" has to be read against
that baseline. `--pool-init-scale` exposes it for the ablation, and only that
knob should change in that experiment (not the FFN).

NOTE: the 0.7097 +/- 0.028 figure quoted here previously used a **shared** encoder
and the literal `[SEP]` separator, so it no longer describes this model. Section 6
re-establishes the number under the current architecture.


In [ ]:
!python src/train_xattn.py --languages EN IT NL --epochs 20 --batch-size 32 --max-len 160 --video-max-len 256 --unfreeze-layers 2 --heads 4 --layers 3 --dropout 0.1 --weight-decay 0.01 --head-lr 1e-3 --fusion gated --warm-start


## 4. Curves

Reads `weights/best_xattn_stereotype_history.csv` written by the cell above.


In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd

h = pd.read_csv('weights/best_xattn_stereotype_history.csv')
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].plot(h['epoch'], h['train_loss'], '--', label='train')
ax[0].plot(h['epoch'], h['val_loss'], '-', label='val')
ax[1].plot(h['epoch'], h['val_macro_f1'], '-o', ms=3, label='val macro-F1')
ax[0].set_title('loss'); ax[1].set_title('validation macro-F1')
for a in ax:
    a.grid(alpha=0.3); a.set_xlabel('epoch'); a.legend()
plt.tight_layout(); plt.show()

best = h.loc[h['val_macro_f1'].idxmax()]
print(f"best epoch {int(best['epoch'])} | macro_f1 {best['val_macro_f1']:.4f} | acc {best['val_acc']:.4f}")


## 5. Confusion matrix + per-language (best checkpoint)

Reloads `weights/best_xattn_stereotype.pt` with the **same token budgets used in
training** (`max-len 160`, `video-max-len 256`), then scores the same
video-grouped validation split (EN/IT/NL only).

The `build_model` call below relies on the current **defaults** (two separate
encoders, `fusion='gated'`, warm-start on). If section 3 was run with a different
encoder mode or `--fusion`, pass the same options here or the state dict will not
load - a different fusion changes `head_in` and adds or removes the gate.


In [ ]:
import numpy as np
import torch
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt

from model_xattn import build_model, DEFAULT_BACKBONE
from model import load_tokenizer
from train import resolve_device
import train_xattn as tx

dev = resolve_device('cuda' if torch.cuda.is_available() else 'cpu')
tok = load_tokenizer(DEFAULT_BACKBONE)
# match section 3's architecture; warm_start is irrelevant here because the state
# dict is loaded right after
net = build_model(DEFAULT_BACKBONE, unfreeze_layers=2, heads=4, layers=3,
                  fusion='gated', warm_start=False)
net.load_state_dict(torch.load('weights/best_xattn_stereotype.pt', map_location=dev))
net.to(dev).eval()

va = va.reset_index(drop=True)
out = tx.collect(net, tx.make_loader(va, tok, 32, 160, 256), dev)
m = tx.metrics_from(out['preds'], out['labels'])
print({k: round(v, 4) for k, v in m.items() if k != 'loss'})

cm = confusion_matrix(out['labels'], out['preds'], labels=[0, 1])
fig, ax = plt.subplots(figsize=(4.4, 4))
ax.imshow(cm, cmap='Blues')
ax.set_xticks([0, 1]); ax.set_xticklabels(['no', 'yes'])
ax.set_yticks([0, 1]); ax.set_yticklabels(['no', 'yes'])
ax.set_title('xattn (best checkpoint)')
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha='center', va='center',
                color='white' if cm[i, j] > cm.max() / 2 else 'black')
plt.tight_layout(); plt.show()

for lang in sorted(va['lang'].unique()):
    sub = va[va['lang'] == lang].reset_index(drop=True)
    o = tx.collect(net, tx.make_loader(sub, tok, 32, 160, 256), dev)
    mm = tx.metrics_from(o['preds'], o['labels'])
    print(f"{lang}: acc {mm['acc']:.4f} | macro_f1 {mm['macro_f1']:.4f}")


## 6. Screen: encoder sharing, the block LR, warm-start and fusion (seed 42)

Each tower now has its **own encoder** by default, so the two writing styles
(terse `title | description` vs. informal, often abusive comments) can specialise
in the blocks that training re-opens. The head input is governed by `--fusion`
(default `gated`) and the decoder blocks are warm-started from the encoder by
default - both are described in section 3.

| name | what it is |
|---|---|
| `sep_u2_l3` | the default: separate encoders, top 2 encoder blocks unfrozen per tower, `--fusion gated`, warm-start ON |
| `sep_u2_l3_none` | default + `--fusion none`: cross-attention is the only channel from the video tower (the previous default) |
| `sep_u2_l3_concat` | default + `--fusion concat`: legacy late fusion, appends `mean-pool(H_v)` to the head (the old `--pooled-v`) |
| `sep_u2_l3_nws` | default + `--no-warm-start`: decoder blocks from random init. Control for the warm-start ablation. |
| `sep_u2_l3_x1e3` | default + `--xattn-lr 1e-3`: decoder blocks at the head LR, i.e. the pre-fix behaviour. The control for Risk 2 (blocks at too big a step). |
| `shared_u2_l3` | default + `--shared-encoder`: one encoder for both towers (the architecture behind every earlier result) |
| `sep_u0_l3` | default + `--unfreeze-layers 0`: both encoders frozen. **Stale - see below.** |

Every row spells out its flags, including `--fusion` and `--warm-start`, so a later
default change cannot silently alter what an already-stored row means.

This is a **single-change screen**: each ablation row differs from `sep_u2_l3` by
exactly one flag. Confirm a candidate on seeds 40/41/42 (the `SEEDS` tuple in the
cell below) before reading anything into it; a one-seed gap under ~2pt is inside the
noise.

Both copies of the encoder start from the same pretrained checkpoint and every
frozen block stays bit-identical between them, so a null on `separate` vs `shared`
means the towers did **not** need to specialise - but reading a null needs a noise
floor smaller than the effect, and see below.

**`sep_u0_l3` must be re-run.** Its `0.6960` was produced while the frozen-encoder
dropout invariant was silently broken for `--unfreeze-layers 0`: all 36 Dropouts in
each frozen tower were in train mode, so a "frozen" encoder was injecting a fresh
random mask every step. The cause is fixed, but the number is not comparable to the
others (it is not even the method it claims to be). The rows with
`--unfreeze-layers 2` - `sep_u2_l3`, `sep_u2_l3_none`, `sep_u2_l3_concat`,
`sep_u2_l3_nws`, `sep_u2_l3_x1e3`, `shared_u2_l3` - were never affected.
`--freeze-epochs` runs were: their phase 1 freezes the encoder entirely, so phase 1
saw the same stochastic "frozen" tower.

**Renaming note (pre-fusion checkpoints).** Any seed-42 checkpoint produced *before*
this change carries the opposite name to its content, and none of them is the new
default (they predate `gated`/warm-start):

```bash
# old files, if still on disk:
mv weights/xattn_sep_u2_l3_s42.pt     weights/xattn_sep_u2_l3_concat_s42.pt   # was WITH pooled_v
mv weights/xattn_sep_u2_l3_npv_s42.pt weights/xattn_sep_u2_l3_none_s42.pt     # was WITHOUT
```

Their numbers were `0.7213` (with pooled_v) and `0.7396` (without), i.e. `+1.83pt`
for dropping it. Against the same-seed, same-flag spread
`0.7366 / 0.7214 / 0.7119 / 0.7217` (~1pt sd, 2.5pt between the extremes) that gap
is **not decidable at n=1**: it is smaller than the distance between two runs of an
*identical* config. Do not quote it as a result. Section 8 is the measurement that
can settle it, and it costs no epochs.

NOTE: every xattn number recorded before this workstream (`u2_l3` = 0.7097 +/-
0.028, etc.) used a shared encoder, the literal `[SEP]` separator **and** the
blocks at the head LR, so those rows are not comparable to these.


In [ ]:
import subprocess
import pandas as pd

SEEDS = (40, 41, 42)  # single-change ablations need >= 3 seeds; report mean +/- sd
BASE = ('--languages EN IT NL --epochs 15 --patience 5 --batch-size 32 '
        '--max-len 160 --video-max-len 256 --dropout 0.1 --weight-decay 0.01 --head-lr 1e-3')
CONFIGS = {
    # default
    'sep_u2_l3':        '--unfreeze-layers 2 --layers 3 --fusion gated  --warm-start',
    # one knob changed per row, relative to the default:
    'sep_u2_l3_none':   '--unfreeze-layers 2 --layers 3 --fusion none   --warm-start',
    'sep_u2_l3_concat': '--unfreeze-layers 2 --layers 3 --fusion concat --warm-start',
    'sep_u2_l3_nws':    '--unfreeze-layers 2 --layers 3 --fusion gated  --no-warm-start',
    'sep_u2_l3_x1e3':   '--unfreeze-layers 2 --layers 3 --fusion gated  --warm-start --xattn-lr 1e-3',
    'shared_u2_l3':     '--unfreeze-layers 2 --layers 3 --fusion gated  --warm-start --shared-encoder',
    'sep_u0_l3':        '--unfreeze-layers 0 --layers 3 --fusion gated  --warm-start',
}

rows = []
for name, flags in CONFIGS.items():
    for s in SEEDS:
        out = f'weights/xattn_{name}_s{s}.pt'
        cmd = f'python src/train_xattn.py --split-seed {s} --seed {s} {BASE} {flags} --out {out}'
        print('RUN:', cmd)
        subprocess.run(cmd, shell=True, check=True)
        h = pd.read_csv(out.replace('.pt', '_history.csv'))
        rows.append({'config': name, 'seed': s,
                     'val_macro_f1': round(float(h['val_macro_f1'].max()), 4)})

df = pd.DataFrame(rows)
df.to_csv('weights/xattn_confirm.csv', index=False)
summary = (df.groupby('config')['val_macro_f1']
             .agg(mean='mean', std='std', n='count').round(4)
             .sort_values('mean', ascending=False))
print(summary.to_string())
print()
print(df.pivot(index='config', columns='seed', values='val_macro_f1').to_string())
print()
print('Claim a difference only if |mean_a - mean_b| is comfortably above the spread.')


## 7. Compare to the prompt baseline (`src/train.py`)

Same seeds and split, but the mmBERT prompt classifier instead of the two-tower
model. The cell below runs it over seeds 40/41/42 and reports mean +/- sd; the old
reference **0.7158** is a single run and is stale - it predates the frozen-dropout
fix that landed in `src/train.py`, so treat it as "re-run me", not as a baseline.

This baseline uses `--unfreeze-layers 2`, so the `--unfreeze-layers 0` failure
section 6 describes never touched it (with `unfreeze_layers >= 1` the ancestor test
already worked). `src/train.py` reapplies `eval_frozen_modules` after
`model.train()`, which is now correct in both cases.

The comparison is only meaningful against the **3-seed** xattn row (`sep_u2_l3`);
a single run of either side is inside the ~1pt spread.


In [ ]:
import os
import subprocess
import pandas as pd

SEEDS = (40, 41, 42)
BASE = ('--languages EN IT NL --epochs 10 --batch-size 32 --max-len 320 '
        '--unfreeze-layers 2 --readout-init random')

rows = []
for s in SEEDS:
    out = f'weights/baseline_s{s}.pt'
    cmd = f'python src/train.py --split-seed {s} --seed {s} {BASE} --out {out}'
    print('RUN:', cmd)
    subprocess.run(cmd, shell=True, check=True)
    h = pd.read_csv(out.replace('.pt', '_history.csv'))
    rows.append({'config': 'prompt_baseline', 'seed': s,
                 'val_macro_f1': round(float(h['val_macro_f1'].max()), 4)})

df = pd.DataFrame(rows)
if os.path.exists('weights/xattn_confirm.csv'):
    df = pd.concat([df, pd.read_csv('weights/xattn_confirm.csv')], ignore_index=True)

summary = (df.groupby('config')['val_macro_f1']
             .agg(mean='mean', std='std', n='count').round(4)
             .sort_values('mean', ascending=False))
print(summary.to_string())


## 8. Fusion ablation: `none` vs `gated` vs `concat`

`--fusion` decides how the video tower reaches the head:

- **`none`** - the head reads `pooled_c`; cross-attention is the **only** channel
  from the video tower (the previous default).
- **`concat`** - append `mean-pool(H_v)`: a *parallel* path that never passes
  through the cross-attention (the old `--pooled-v`).
- **`gated`** (default) - add `sigmoid(W.[pooled_c ; mean-pool(H_v)]) *
  video_proj(mean-pool(H_v))`: a learned, initially mostly-closed gate on that same
  parallel path.

Three tests, because they answer different questions:

1. **Train-time** (rows `sep_u2_l3` / `sep_u2_l3_none` / `sep_u2_l3_concat` in
   section 6). Retrain with each fusion mode across seeds 40/41/42, report mean and
   spread. No single-run gap of this size settles it.
2. **Inference-time** (`src/diag_pooled_v.py`, no retraining, no second run). For a
   **`concat`** model, re-run *only the head* with the pooled-v half of its input
   zeroed or permuted. The encoders are not re-run, so this is exact and free.
   Zeroing asks whether the path is load-bearing at all; the permutations keep
   `pooled_v`'s marginal distribution and only break its pairing with the comment,
   so a drop there means the head uses the *content* rather than merely tolerating
   the extra inputs. Section B is **not available for `gated`/`none`** (their head
   input is `d`, not `2d`); for `gated` the gate's own magnitude is the readout.
3. **Input-level perturbation** (section E). Changes the video *input* - permutes
   videos within/across languages, or replaces their content - and re-runs the whole
   model. This is the test that speaks to *reliance* on the paired video.

The script also prints the two numbers the hypothesis makes a prediction about:

- the **cross-attention entropy** over the video keys - if a fusion shortcut is what
  lets the attention stay diffuse, removing it should force sharper attention;
- a **linear probe on the frozen `pooled_c`** - if the cross-attention has to carry
  more of the label signal, the fused comment state should become *more*
  informative on its own.

Pre-registered prediction (from the `none`-vs-`concat` question, before the runs):
**entropy down, probe up** once the parallel video term is removed.

If all three (metric, entropy, probe) are flat, the hypothesis that the parallel
term suppresses the cross-attention is dead, and the honest reading is "dead weight
the head learned to ignore" rather than "shortcut that stole the attention's job".
If zeroing it *does* cost real macro-F1 while the train-time gap stays inside the
noise, then dropping it is a trade rather than a free win. Both directions are
reported, so this section cannot be read as confirmatory on either side.


In [ ]:
import os
import subprocess

# One checkpoint per fusion mode, all from section 6 (seed 42).
# Section B (head-only zeroing of pooled_v) only exists for `concat`; `gated` and
# `none` have a d-dim head input, so the script reports it as n/a for them.
PAIRS = {
    'concat (--fusion concat)':          'weights/xattn_sep_u2_l3_concat_s42.pt',
    'gated  (--fusion gated, default)':  'weights/xattn_sep_u2_l3_s42.pt',
    'none   (--fusion none)':            'weights/xattn_sep_u2_l3_none_s42.pt',
}

for label, ckpt in PAIRS.items():
    if not os.path.exists(ckpt):
        print(f'SKIP {label}: {ckpt} not found - run section 6 first')
        continue
    print('=' * 72)
    print(label, '->', ckpt)
    print('=' * 72)
    subprocess.run(f'python src/diag_pooled_v.py --checkpoint {ckpt} '
                   f'--languages EN IT NL', shell=True, check=False)

print()
print('Read section B of the concat checkpoint (zeroing pooled_v at inference), then')
print('compare section C (entropy, top-1 weight) and D (probes) across the three.')
print('Pre-registered prediction: entropy DOWN and the pooled_c probe UP once the')
print('parallel video term is removed.')
print('Section E is the reliance test: on a trained checkpoint, permuting or emptying')
print('the video input should cost macro-F1 if the model actually uses the paired video.')


### Without a notebook

```bash
git clone https://github.com/AmnO-O/Try.git && cd Try
pip install -r requirements.txt
export LGBT_DATA_DIR=/path/to/tsvs            # or put them in ./LGBT/

# default config: MiniLM + 3 decoder blocks, top 2 blocks trainable, gated fusion,
# warm-started decoder (backbone 2e-5, decoder blocks 1e-4, head 1e-3 - section 3)
python src/train_xattn.py --languages EN IT NL --epochs 20 --batch-size 32 \
    --max-len 160 --video-max-len 256 --unfreeze-layers 2 --layers 3 \
    --fusion gated --warm-start

# --- the two new knobs, each as a single change against the default ---

# fusion: none = cross-attention only (previous default); concat = legacy late fusion
python src/train_xattn.py --languages EN IT NL --unfreeze-layers 2 --layers 3 \
    --fusion none
python src/train_xattn.py --languages EN IT NL --unfreeze-layers 2 --layers 3 \
    --fusion concat

# warm-start off: decoder blocks from random init (the control for --warm-start)
python src/train_xattn.py --languages EN IT NL --unfreeze-layers 2 --layers 3 \
    --fusion gated --no-warm-start

# decoder blocks at the head LR (pre-fix) / a softer step / frozen-dropout on
python src/train_xattn.py --languages EN IT NL --xattn-lr 1e-3 --unfreeze-layers 2 --layers 3
python src/train_xattn.py --languages EN IT NL --xattn-lr 2e-4 --unfreeze-layers 2 --layers 3
python src/train_xattn.py --languages EN IT NL --keep-frozen-dropout --unfreeze-layers 2 --layers 3

# two-phase: encoder frozen for 5 epochs, then unfreeze the top 2 blocks
python src/train_xattn.py --languages EN IT NL --freeze-epochs 5 \
    --unfreeze-layers 2 --layers 3

# move the title into the query: comment </s> title, K/V = description
python src/train_xattn.py --languages EN IT NL --query-fields comment,title \
    --unfreeze-layers 2 --layers 3

# same fields, title first (order follows the flag)
python src/train_xattn.py --languages EN IT NL --query-fields title,comment \
    --unfreeze-layers 2 --layers 3

# title in the query AND kept in K/V (query `title | comment`, K/V `title | description`)
python src/train_xattn.py --languages EN IT NL --query-fields title,comment \
    --keep-kv-title --unfreeze-layers 2 --layers 3

# ... and the matching diagnostics on any checkpoint: fusion-mode inference, the
# concat head-only ablation, cross-attention entropy, and linear probes (section 8)
python src/diag_pooled_v.py --checkpoint weights/xattn_sep_u2_l3_s42.pt

# invariant test: no Dropout inside a frozen subtree may be in train mode.
# Exit code 0 = holds for unfreeze 0 and >=1, both keep-frozen-dropout values,
# both encoder modes, the --freeze-epochs transition, and the prompt baseline
python src/probe_frozen_state.py

# numerical invariant: an all-padding row may not produce a NaN, in the forward
# pass or in the backward pass; the data layer refuses such a row up front and the
# model boundary refuses an all-padded video (the diagnostic path is NaN otherwise)
python src/probe_allpad.py

# does an encoder actually re-open its LAST n blocks? Both backbones, expected
# set taken from the module tree, plus negative/over-range values that must raise
python src/probe_unfreeze.py

# the fusion stack is a decoder: self-attn is full (not causal) and mixes comment
# tokens, padding masks are honoured on both attentions, and an all-padded comment
# row stays finite in forward and backward
python src/probe_decoder.py

# the prompt baseline extracts the pretrained head correctly on BERT / RoBERTa /
# ModernBERT (the BERT container hop and the RoBERTa residual were both wrong)
python src/probe_extract_parts.py

# the attention pool starts uniform; initial entropy at several pool scales
python src/probe_pool_init.py

# parameter footprint of one vs two towers, per backbone
python src/report_cost.py

# tighter attention-pool init (default 0.02; the pool starts at mean-pooling)
# change ONLY this in the pool-init ablation, not the FFN
python src/train_xattn.py --languages EN IT NL --pool-init-scale 0.5 \
    --unfreeze-layers 2 --layers 3

# share one encoder between the towers (previous architecture; default is separate)
python src/train_xattn.py --languages EN IT NL --shared-encoder \
    --unfreeze-layers 2 --layers 3

# prompt baseline on the same split (EN/IT/NL)
python src/train.py --languages EN IT NL --epochs 10 --batch-size 32 \
    --max-len 320 --unfreeze-layers 2 --readout-init random

# swap the encoder / deepen the decoder
python src/train_xattn.py --backbone jhu-clsp/mmbert-base --layers 4

# augmented train split (built with src/augment_roundtrip.py)
python src/train_xattn.py --languages EN IT NL \
    --train-extras aug_rt_IT_to_EN_training.tsv,aug_rt_NL_to_EN_training.tsv
```

Checkpoints and history CSVs land in `weights/`.
